In [1]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

In [1]:
import numpy as np 
import qiskit_metal as metal
from qiskit_metal import draw, Dict 
from qiskit_metal.qlibrary.core import QComponent 

In [2]:
%reload_ext autoreload 
%autoreload 2 

In [3]:
# Function that rotates points by a certain degree. 
# This will be used to keep component nodes sticking to the same point 
# Even while undergoing a rotation from defining an `orientation` option. 

def rotate_point(point, angle):
    """
    Rotates a 2D point about the origin.
    
    Args:
        point (array-like): The (x, y) coordinates
        angle (float): The rotation angle in radians
    
    Returns:
        np.array: rotated point
        
    """
    
    rot_matrix = np.array([[np.cos(angle), -np.sin(angle)],
                           [np.sin(angle),  np.cos(angle)]])
    return np.dot(rot_matrix, point)

In [4]:
design = metal.designs.DesignPlanar() 
gui = metal.MetalGUI(design) 
design.overwrite_enabled = True 

07:14PM 46s CRITICAL [_qt_message_handler]: line: 0, func: None(), file: None  WARNING: Populating font family aliases took 196 ms. Replace uses of missing font family "Courier" with one that exists to avoid this cost. 



In [6]:
%metal_heading Library Components 

In [7]:
%metal_print Alignment Square

In [10]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

""" Bandage. 

.. code-block:: 

    _________________________
   |                         |
   |                         |
   |                         | 
   |                         |
   |            *            |
   |                         |
   |                         |  
   |                         |
   |                         |
    ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾

"""

import numpy as np 
from qiskit_metal import draw, Dict 
from qiskit_metal.qlibrary.core import QComponent 

class AlignmentMarker(QComponent): 
    """ 
    Inherits `QComponent` class. 

    Description: 
        A square alignment marker for the Raith. 

    Options: 
        * size    : The side length of the alignment marker 
        * pos_x/_y: position of the bandage on chip
        * orientation : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise 
        * layer   : the layer number for the bandage

    """ 

    # Default drawing options 
    default_options = Dict(
        size='200um',
        pos_x='0um', 
        pos_y='0um', 
        orientation='0', 
        layer='1', 
    )

    # Component metadata
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='alignment_marker', 
        _qgeometry_table_poly='True', 
    )

    def make(self):
        """ Converts self.options into QGeometry. """ 
        p = self.parse_options() 

        marker = draw.rectangle(p.size, p.size) 
        marker = draw.rotate(marker, p.orientation)
        marker = draw.translate(marker, p.pos_x, p.pos_y) 

        self.add_qgeometry('poly', {'AlignmentMarker': marker}, layer=p.layer)

        # Node doesn't change under rotation 
        # so just set the origin node as pos_x/_y location 
        nodes = Dict() 
        nodes.origin = np.array((p.pos_x, p.pos_y)) 

        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None) 

In [13]:
%metal_print Claw 

In [11]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

""" Bandage. 

.. code-block:: 


                `finger0`   `finger1` 
                    /          / 
               |‾‾‾/‾|    |‾‾‾/‾| 
               | |‾| |    | |‾| | 
               | | | |    | | | | 
               | | | |    | | | | 
               | | |  ‾/‾‾  | | | 
               | |  ‾‾//‾‾‾‾  | | 
               |  ‾‾‾//  |‾‾‾‾  | 
                ‾‾‾‾//| *| |‾‾‾‾
                   // |  | | 
   `pad_cutout`__ //| |  | | 
                  /  ‾ /  ‾
         `pad` __/    /
                     /
            `cpw` __/
"""

import numpy as np 
from qiskit_metal import draw, Dict 
from qiskit_metal.qlibrary.core import QComponent 

class Claw(QComponent): 
    """ 
    Inherits `QComponent` class. 

    Description: 
        A claw-shaped coupling pad.  

    Options: 
        * cpw_width: The width of the CPW trace 
        * cpw_gap  : The CPW gap of the connected trace 
        * base_length: The length (`x`) of the base segment 
        * base_width : The width (`y`) of the base segment 
        * finger_length: The length (`y`) of the finger measured w.r.t bottom of base 
        * finger_width : The width (`x`) of the fingers 
        * gap: The substrate gap between the coupling pad and the ground plane 
        * cpw_length   : The length of the CPW as measured from the origin (*)
        * pos_x/_y: position of the bandage on chip
        * orientation : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise 
        * layer   : the layer number for the bandage

    """ 

    # Default drawing options 
    default_options = Dict(
        cpw_width='20um', 
        cpw_gap='10um', 
        base_length='100um', 
        base_width='20um',
        finger_length='100um', 
        finger_width='20um', 
        gap='5um', 
        cpw_length='30um', 
        pos_x='0um', 
        pos_y='0um', 
        orientation='0', 
        layer='1', 
    )

    # Component metadata
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='Claw', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Convert self.options into QGeometry. """ 
        p = self.parse_options() 
        nodes = Dict() 

        # Base section 
        base = draw.rectangle(p.base_length, p.base_width) 
        base = draw.translate(base, 0, p.base_width/2) 

        base_cutout = draw.rectangle(p.base_length + 2*p.cpw_gap, p.base_width + 2*p.cpw_gap) 
        base_cutout = draw.translate(base_cutout, 0, p.base_width/2)

        # Left and right fingers 
        finger = draw.rectangle(p.finger_width, p.finger_length) 
        l_fing = draw.translate(finger, (-p.base_length+p.finger_width)/2, p.finger_length/2)
        r_fing = draw.translate(finger, (+p.base_length-p.finger_width)/2, p.finger_length/2)

        finger_cutout = draw.rectangle(p.finger_width + 2*p.cpw_gap, p.finger_length + 2*p.cpw_gap) 
        l_fing_cutout = draw.translate(finger_cutout, (-p.base_length+p.finger_width)/2, p.finger_length/2) 
        r_fing_cutout = draw.translate(finger_cutout, (+p.base_length-p.finger_width)/2, p.finger_length/2)

        # Bottom connecting CPW section 
        cpw_rect = draw.rectangle(p.cpw_width, p.cpw_length) 
        cpw_rect = draw.translate(cpw_rect, 0, -p.cpw_length/2) 

        cpw_cutout = draw.rectangle(p.cpw_width + p.cpw_gap*2, p.cpw_length) 
        cpw_cutout = draw.translate(cpw_cutout, 0, -p.cpw_length/2) 

        # Performing rotations and translations 
        geom_list = [
            base, l_fing, r_fing, cpw_rect, 
            base_cutout, l_fing_cutout, r_fing_cutout, cpw_cutout, 
        ] 
        geom_list = draw.rotate(geom_list, p.orientation, origin=(0,0)) 
        geom_list = draw.translate(geom_list, p.pos_x, p.pos_y) 
        [
            base, l_fing, r_fing, cpw_rect, base_cutout, 
            l_fing_cutout, r_fing_cutout, cpw_cutout,
        ] = geom_list 

        # Finishing component 
        claw = draw.union([base, l_fing, r_fing, cpw_rect])
        cutout = draw.union([base_cutout, l_fing_cutout, r_fing_cutout, cpw_cutout]) 
        claw = draw.subtract(cutout, claw)

        # Converting to QGeometry
        self.add_qgeometry('poly', {'Claw': claw}, layer=p.layer) 

        # Positioning nodes 
        nodes.origin = np.zeros(2) 
        nodes.pad = nodes.origin + [0, p.base_width] 
        nodes.pad_cutout = nodes.pad + [0, p.cpw_gap] 
        nodes.cpw = nodes.origin + [0, -max(p.cpw_length, p.cpw_gap)] 
        nodes.finger1 = nodes.origin + [+(p.base_length-p.finger_width)/2, p.finger_length] 
        nodes.finger0 = np.array((-nodes.finger1[0], nodes.finger1[1]))

        # Moving nodes along with component  
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        theta = np.deg2rad(p.orientation) 

        for key, point in nodes.items():
            rotated = rotate_point(point, theta) 
            nodes[key] = rotated + translation_vec

        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None)

In [29]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

""" Bandage. 

.. code-block:: 



""" 

import numpy as np 
from qiskit_metal import draw, Dict 
from qiskit_metal.qlibrary.core import QComponent 

class CouplingPad(QComponent): 
    """ 
    Inherits `QComponent` Class. 

    Description: 
        A coupling pad. 

    Options: 
        * cpw_width : The width of the CPW trace connected to the coupling pad 
        * cpw.gap   : The CPW gap of the trace connected to the coupling pad 
        * pad_length: The length (`y`) of the coupling pad 
        * pad_width : The width (`x`) of the coupling pad 
        * gap: The gap between the coupling pad and the ground plane 
        * cpw_length: The length of the CPW as measured from the origin 
        * pos_x/_y: position of the bandage on chip
        * orientation : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise 
        * layer   : the layer number for the bandage

    """ 

    # Default drawing options 
    default_options = Dict(
        cpw_width='20um', 
        cpw_gap='10um', 
        pad_length='300um', 
        pad_width='20um',
        gap='5um', 
        cpw_length='50um', 
        pos_x='0um', 
        pos_y='0um', 
        orientation='0', 
        layer='1', 
    )

    # Component metadata
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='CouplingPad', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Converts self.options into QGeometry. """ 
        p = self.parse_options() 
        nodes = Dict() 

        # Encompassing cutout 
        cutout = draw.rectangle(p.pad_length + 2*p.gap, p.pad_width + 2*p.gap)
        cutout = draw.translate(cutout, 0, +p.pad_width/2) 

        # Inner pad gap 
        pad = draw.rectangle(p.pad_length, p.pad_width) 
        pad = draw.translate(pad, 0, +p.pad_width/2) 

        # Encompassing cpw 
        cpw_cutout = draw.rectangle(p.cpw_width + 2*p.gap, p.cpw_length) 
        cpw_cutout = draw.translate(cpw_cutout, 0, -p.cpw_length/2)

        # CPW gap, 1e-10 to avoid rendering issues
        cpw = draw.rectangle(p.cpw_width, p.cpw_length+1e-10) 
        cpw = draw.translate(cpw, 0, -p.cpw_length/2) 

        # Translations \& Rotations 
        geom_list = [cutout, pad, cpw_cutout, cpw] 
        geom_list = draw.rotate(geom_list, p.orientation, origin=(0, 0)) 
        geom_list = draw.translate(geom_list, p.pos_x, p.pos_y) 
        [cutout, pad, cpw_cutout, cpw] = geom_list 

        # Combining components 
        coupling_pad = draw.union([pad, cpw]) 
        coupling_cut = draw.union([cutout, cpw_cutout]) 
        coupling_pad = draw.subtract(coupling_cut, coupling_pad) 

        # Converting to QGeometry
        self.add_qgeometry('poly', {'CouplingPad': coupling_pad}, layer=p.layer) 

        # Positioning nodes 
        nodes.origin = np.zeros(2) 
        nodes.cpw = nodes.origin + [0, -max(p.gap, p.cpw_length)] 
        nodes.pad = nodes.origin + [0, p.pad_width] 

        # Moving nodes along with component  
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        theta = np.deg2rad(p.orientation) 

        for key, point in nodes.items():
            rotated = rotate_point(point, theta) 
            nodes[key] = rotated + translation_vec

        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None) 